TESTING CLASS

In [ ]:
from HomeMessagesDB import HomeMessagesDB
db = HomeMessagesDB("sqlite:///my_database.sqlite")
db.create_db()


In [ ]:
db.drop_table("smartthings")

In [ ]:
db.insert_table_smartthings("data/smartthings/smartthings.20230107.tsv.gz")

CONVERT TIME

In [ ]:
# Ensure we're working on a copy to avoid warnings
filt = filt.copy()

# Convert to datetime and then to UNIX timestamp 
filt["epoch"] = pd.to_datetime(filt["epoch"], utc=True).astype("int64") // 10**9


In [ ]:
filt["epoch"] = pd.to_datetime(filt["epoch"], utc=True)
filt.loc[:,"epoch"] = filt["epoch"].astype("int64") // 10**9

In [ ]:
filt

In [ ]:
#pd.to_datetime(filt["epoch"], utc=True, inplace = True)
pd.to_datetime(filt.epoch)


In [ ]:
filt["epoch"] = pd.to_datetime(filt["epoch"], utc=True)

In [ ]:
#filt.epoch.timestamp()

In [ ]:
from datetime import datetime
#datetime.timestamp(filt.epoch)


In [ ]:

filt.loc[:,"epoch"] = filt["epoch"].astype("int64") // 10**9


In [ ]:
filt

CODES FOR CREATING TABLES

In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS devices (
   name TEXT PRIMARY KEY,
   loc TEXT NOT NULL,
   level TEXT NOT NULL,
   PRIMARY KEY (name)
   FOREIGN KEY (name) 
      REFERENCES smartthings (name)
)

In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS smartthings (
   name TEXT NOT NULL,
   epoch TEXT NOT NULL,
   capability TEXT NOT NULL,
   attribute TEXT NOT NULL,
   value TEXT NOT NULL,
   unit TEXT,
   PRIMARY KEY (name,epoch,attribute)
   FOREIGN KEY (name) 
      REFERENCES devices (name)
)

In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS p1e (
   epoch INTEGER PRIMARY KEY,
   ImportT1kWh FLOAT,
   ImportT2kWh FLOAT,
   ExportT1kWh FLOAT,
   ExportT2kWh FLOAT,
   PRIMARY KEY (epoch)
   FOREIGN KEY (epoch) 
      REFERENCES smartthings (epoch)
)

In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS p1g (
   epoch INTEGER PRIMARY KEY,
   Total_gas_used FLOAT,
   PRIMARY KEY (epoch)
   FOREIGN KEY (epoch) 
      REFERENCES smartthings (epoch)
)

Exploring Data to create Dataset

In [ ]:
import pandas as pd
import glob

file_list = glob.glob('data/smartthings/smartthings.*')

# Read and concatenate all matching files into a single DataFrame
smart_things = pd.concat([pd.read_csv(f, sep='\t') for f in file_list], ignore_index=True)

In [ ]:
filt = smart_things[~smart_things.duplicated()]


In [ ]:
filt.reset_index(inplace = True)

In [ ]:
filt

In [ ]:

#filt[isinstance(filt.value,str)]
for val in filt.value:
    i = 1 if isinstance(val,str) else 0


In [ ]:
filt['is_str'] = filt['value'].apply(lambda x: 1 if isinstance(x, str) else 0)


In [ ]:
for i in filt.index:
    try:
        val = float(filt.loc[i, "value"])
        filt.loc[i, "int"] = val
    except:
        val = filt.loc[i, "value"]
        filt.loc[i, "str"] = val



In [ ]:
# Attempt to convert 'value' to numeric (invalid parsing will produce NaN)
numeric_values = pd.to_numeric(filt["value"], errors="coerce")

# Create 'int' column: will be float values or NaN
filt["int"] = numeric_values

# Create 'str' column: keep original values only where conversion failed
filt["str"] = filt["value"].where(numeric_values.isna())


In [ ]:
filt = filt.copy()

filt.loc[:, 'int'] = pd.to_numeric(filt['value'], errors='coerce')
filt.loc[:, 'str'] = filt['value'].where(filt['int'].isna())

In [ ]:
filt["name"].unique()

In [ ]:
pd.read_csv("data/P1e/P1e-2022-01-01-2022-05-07.csv.gz")

In [ ]:
filt.iloc[4, ]

In [ ]:
# Of duplicated entries
smart_things.shape[0]-filt.shape[0]

In [ ]:
#filt[filt.duplicated(subset=['name', 'epoch', 'attribute'])]

#filt.unique(nam)
#df1.groupby(['A','B']).size()

unique_pairs = filt['value'].drop_duplicates()
unique_pairs[isinstance(unique_pairs.value,str)]


In [ ]:
filt[filt.epoch.duplicated() & filt.name.duplicated()]

In [ ]:
import calendar
#calendar.timegm(filt.epoch)

calendar.timegm(pd.to_datetime(filt['epoch']))

In [ ]:
filt.epoch.dtype

Create Database

In [ ]:
%load_ext sql
%sql sqlite:///energy.db

%%sql
CREATE TABLE IF NOT EXISTS 


In [ ]:
import sqlalchemy as sa
sa.create_engine("sqlite:///energy.db")

In [ ]:
file_list = glob.glob('data/P1e/*.csv') + glob.glob('data/P1e/*.csv.gz')

# Read and concatenate all matching files into a single DataFrame
p1e = pd.concat([pd.read_csv(f) for f in file_list], ignore_index=True)


In [ ]:
p1e

In [ ]:
for column in p1e:
        p1e.rename(columns = {column : column.replace(" ", "_")}, inplace = True)

In [ ]:
p1e

In [ ]:
file_list = glob.glob('data/P1g/*.csv') + glob.glob('data/P1g/*.csv.gz')

# Read and concatenate all matching files into a single DataFrame
p1g = pd.concat([pd.read_csv(f) for f in file_list], ignore_index=True)

In [ ]:
p1g

In [ ]:
import logging
try:
    float("str")
except Exception as e:
    logging.error(f"Pandas could not insert table  in the database: {e}")
    raise e